In [ ]:
import os
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns

from tqdm.notebook import tqdm
from pyphylon.util import load_config

In [ ]:
mlst = pd.read_csv('/mnt/craig/pan_phylon/P_aeruginosa/data/processed_ncbi/mlst/ncbi_mlst.tsv', sep='\t', header=None, dtype='object')

# Add column names
mlst.columns = [
    'genome_id',
    'schema',
    'mlst',
    'allele1',
    'allele2',
    'allele3',
    'allele4',
    'allele5',
    'allele6',
    'allele7']

mlst['genome_id'] = mlst['genome_id'].str.split('/', expand=True)[9].str.replace('.fna', '')
mlst

# Enrich metadata

For now, its just MLST. Add in other things as needed

In [ ]:
mash_scrubbed_metadata = pd.read_csv('/mnt/craig/pan_phylon/P_aeruginosa/data/interim/2b_ncbi_mash_scrubbed_species_metadata.csv', index_col=0, dtype='object')

display(
    mash_scrubbed_metadata.shape,
    mash_scrubbed_metadata.head()
)

In [ ]:
mlst = mlst.set_index('genome_id')
mlst

In [ ]:
mash_scrubbed_metadata['mlst_source'] = '-'
mash_scrubbed_metadata['mlst'] = '-'

In [ ]:
mash_scrubbed_metadata = mash_scrubbed_metadata.set_index('genome_id')

In [ ]:
for g_id in mlst.index.values:
    #add in all mlst values from PUBMLST, wherever the value is not missing ('-')
    if mlst.loc[g_id, 'mlst'] != '-':
        if g_id in mash_scrubbed_metadata.index.values:
            mash_scrubbed_metadata.loc[g_id, 'mlst'] = mlst.loc[g_id, 'mlst']
            mash_scrubbed_metadata.loc[g_id, 'mlst_source'] = 'PubMLST'

In [ ]:
mash_scrubbed_metadata[['mlst','mlst_source']]

## gap filling using BVBRC annotations

In [ ]:
mash_scrubbed_metadata = mash_scrubbed_metadata.reset_index().set_index('genbank_accessions')
mash_scrubbed_metadata

In [ ]:
# download metadata from BV-BRC
bvbrc = pd.read_csv('PA_genome_metadata_bvbrc.csv')
bvbrc.head()

In [ ]:
bvbrc = bvbrc.dropna(subset=['Assembly Accession','MLST'])
bvbrc = bvbrc.set_index('Assembly Accession')

In [ ]:
# current
mash_scrubbed_metadata.mlst_source.value_counts()

In [ ]:
# making sure there arent different mlst assignments for genomes with the same accession
for idx in mash_scrubbed_metadata.index.values:
    if idx in bvbrc.index.values:
        if type(bvbrc.loc[idx, 'MLST']) != str:
            if len(np.unique([el.rpartition('.')[-1] for el in bvbrc.loc[idx, 'MLST'].values])) > 1:
                print(idx)

In [ ]:
# gap filling
for idx in mash_scrubbed_metadata.index.values:
    if mash_scrubbed_metadata.loc[idx, 'mlst'] == '-':
        if idx in bvbrc.index.values:
            if type(bvbrc.loc[idx, 'MLST']) == str:
                mash_scrubbed_metadata.loc[idx, 'mlst'] = bvbrc.loc[idx, 'MLST'].rpartition('.')[-1]
            else:
                mash_scrubbed_metadata.loc[idx, 'mlst'] = bvbrc.loc[idx, 'MLST'][0].rpartition('.')[-1]
            mash_scrubbed_metadata.loc[idx, 'mlst_source'] = 'BVBRC'

In [ ]:
# after gap filling ( we dont want the PubMLST count to change)
mash_scrubbed_metadata.mlst_source.value_counts()

In [ ]:
mash_scrubbed_metadata = mash_scrubbed_metadata.reset_index()

In [ ]:
mash_scrubbed_metadata.mlst.value_counts()